### 🎯 **서비스 목표: 논문 요약 LLM 서비스**

#### ✔️ 구성
- **입력**:
  - 논문 PDF: [https://arxiv.org/pdf/2005.11401](https://arxiv.org/pdf/2005.11401)
  - Prompt: 사용자가 직접 구성
- **방법론**:
  - LLM: `gpt-4o-mini`
  - RAG 기반 PDF 처리 (LangChain 사용)
- **출력**:
  - LLM의 응답 그대로 출력

---

### ✅ 요구사항 체크리스트
- [x] 논문 PDF를 docs로 변환 (PDF Loader 활용)
- [x] 요약을 위한 프롬프트 설계
- [x] 요약 생성 (LLM + RAG)

---


#### 0. 라이브러리 세팅
---


In [ ]:
# 🛠️ 패키지 설치
!pip install langchain langchain-community langchain-openai langchainhub chromadb langchain-chroma pypdf requests PyMuPDF


In [ ]:

# 🔐 .env에서 API 키 불러오기
from dotenv import load_dotenv
import os

# openai_key = ""
load_dotenv()  # .env 파일 로드
openai_key = os.getenv("OPENAI_API_KEY")  # 키 가져오기



# 📚 라이브러리 임포트
from langchain.docstore.document import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain import hub

In [ ]:
Document

langchain_core.documents.base.Document

In [ ]:
RecursiveCharacterTextSplitter

langchain_text_splitters.character.RecursiveCharacterTextSplitter

In [ ]:
Chroma

langchain_chroma.vectorstores.Chroma

In [ ]:
hub.BasePromptTemplate

langchain_core.prompts.base.BasePromptTemplate

#### 1. PDF를 LangChain으로 로드 (예: `PyPDFLoader` 사용)
---
서비스 구현 코드 (LangChain + gpt-4o-mini + RAG)

In [ ]:
import requests
import fitz  # PyMuPDF

pdf_url = "https://arxiv.org/pdf/2005.11401"
response = requests.get(pdf_url)
pdf_data = response.content

# PDF 파싱
doc = fitz.open(stream=pdf_data, filetype="pdf")
full_text = ""
for page in doc:
    full_text += page.get_text()

# LangChain 문서 형식으로 변환
langchain_doc = [Document(page_content=full_text)]

# 문서 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = splitter.split_documents(langchain_doc)


In [ ]:
pdf_data[:100]

b'%PDF-1.5\n%\x8f\n172 0 obj\n<< /Filter /FlateDecode /Length 4144 >>\nstream\nx\xda\x95:\xc9\x92\xdbF\xb2w}\x05//\x1e\x18OD\xa36,\x9e\x93\xc6\xb2<\xf6XcI\xee'

In [ ]:
langchain_doc[0]

Document(metadata={}, page_content='Retrieval-Augmented Generation for\nKnowledge-Intensive NLP Tasks\nPatrick Lewis†‡, Ethan Perez⋆,\nAleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,\nMike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†\n†Facebook AI Research; ‡University College London; ⋆New York University;\nplewis@fb.com\nAbstract\nLarge pre-trained language models have been shown to store factual knowledge\nin their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-\nstream NLP tasks. However, their ability to access and precisely manipulate knowl-\nedge is still limited, and hence on knowledge-intensive tasks, their performance\nlags behind task-speciﬁc architectures. Additionally, providing provenance for their\ndecisions and updating their world knowledge remain open research problems. Pre-\ntrained models with a differentiable access mechanism to explicit non-parametric\nmemory have so far

In [ ]:
splits

[Document(metadata={}, page_content='Retrieval-Augmented Generation for\nKnowledge-Intensive NLP Tasks\nPatrick Lewis†‡, Ethan Perez⋆,\nAleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,\nMike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†\n†Facebook AI Research; ‡University College London; ⋆New York University;\nplewis@fb.com\nAbstract\nLarge pre-trained language models have been shown to store factual knowledge\nin their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-\nstream NLP tasks. However, their ability to access and precisely manipulate knowl-\nedge is still limited, and hence on knowledge-intensive tasks, their performance\nlags behind task-speciﬁc architectures. Additionally, providing provenance for their\ndecisions and updating their world knowledge remain open research problems. Pre-\ntrained models with a differentiable access mechanism to explicit non-parametric'),
 Document(metad

#### 2. 요약 prompt 구성
---

In [ ]:
# 요약 요청용 질문 프롬프트
input_prompt = """
다음 논문을 아래의 템플릿 형식에 따라 요약해줘. 각 항목은 간결하면서도 중요한 내용을 포함해줘:

1. 📌 제목 요약 (Title Summary):
2. 🧪 연구 목적 (Research Objective):
3. 🛠 방법론 요약 (Methodology):
4. 🔬 주요 실험 결과 (Key Findings):
5. 🧠 기여 및 의의 (Contribution):
6. 🌍 응용 가능성 (Applications):

출력은 위 순서와 형식을 유지해줘. 한국어로 작성해줘.
"""


# 문서 내용을 하나의 문자열로 포맷
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


#### 3. RAG 기반 파이프라인 완성
---


In [ ]:
# 벡터스토어 생성 및 검색기 구성
vectorstore = Chroma.from_documents(splits, OpenAIEmbeddings(api_key=openai_key))
retriever = vectorstore.as_retriever()

# 프롬프트 템플릿 로드
rag_prompt = hub.pull("rlm/rag-prompt")

# 검색 → context로 구성된 프롬프트 생성
retrieved_docs = retriever.invoke(input_prompt)
final_prompt = rag_prompt.invoke({
    "context": format_docs(retrieved_docs),
    "question": input_prompt
})


/usr/local/lib/python3.11/dist-packages/langsmith/client.py:280: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


#### 4. 결과 출력
---

In [ ]:
# LLM 호출 (gpt-4o-mini)
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_key)
response = llm.invoke(final_prompt)

# 출력
print("\n📄 논문 요약 결과:")
print(response.content)



📄 논문 요약 결과:
1. 📌 제목 요약 (Title Summary): 검색 엔진을 이용한 신경망 기계 번역  
2. 🧪 연구 목적 (Research Objective): 본 연구는 검색 엔진의 정보를 활용하여 기계 번역 성능을 향상시키는 것을 목표로 한다.  
3. 🛠 방법론 요약 (Methodology): 검색된 정보를 신경망 번역 모델의 입력으로 통합하여 문맥을 개선하고 번역 품질을 높인다.  
4. 🔬 주요 실험 결과 (Key Findings): 제안된 방법은 기존 기계 번역 시스템보다 높은 정확도를 보였으며, 특히 자주 쓰이는 구문에 대한 번역에서 효과적이었다.  
5. 🧠 기여 및 의의 (Contribution): 검색 엔진의 활용을 통해 기계 번역의 성능을 개선하는 새로운 접근 방식을 제시하였다.  
6. 🌍 응용 가능성 (Applications): 이 연구는 다국어 번역 시스템, 실시간 커뮤니케이션 도구 등 다양한 언어 처리 애플리케이션에 적용될 수 있다.



---

## ❌ 요약의 핵심 문제:  
**"논문 내용을 요약한 것이 아니라, 논문을 오해해서 엉뚱한 주제로 요약하고 있음"**

---

## 📊 항목별 개선 분석

| 항목 | 현재 답변 내용 | ❌ 문제점 | ✅ 개선 방향 |
|------|----------------|----------|--------------|
| **1. 제목 요약** | 검색 엔진을 이용한 신경망 기계 번역 | 논문 제목은 **“Retrieval-Augmented Generation”**인데 번역이라는 단어는 없음 | RAG 개념을 반영한, 예: *검색 기반 언어 생성 모델* |
| **2. 연구 목적** | 검색 정보를 활용해 번역 성능 향상 | 이 논문은 **기계 번역이 아니라 knowledge-intensive tasks** (예: open-domain QA)을 목표로 함 | “외부 지식을 활용해 생성 모델의 정밀도·팩트성 향상” 등으로 수정 필요 |
| **3. 방법론 요약** | 검색 정보 → 입력 보강 → 번역 개선 | 구조 설명은 그럴듯하지만 **실제로는 BART+Retriever 구조**로, 기계 번역용이 아님 | RAG-Token, RAG-Sequence의 구조적 차이와 Dense Retriever 사용 등을 요약해야 함 |
| **4. 실험 결과** | 번역 정확도 향상 | 논문은 번역을 실험하지 않았음. **QA, NLG, fact verification, Jeopardy question gen** 등 실험 | 실제 실험 대상과 향상된 성능을 정확히 설명해야 함 |
| **5. 기여 및 의의** | 번역 성능 개선의 새로운 접근 | ❌ 논문은 번역과 관련 없음 | “비지도 학습 기반의 retrieval + generation 통합 구조 제시” |
| **6. 응용 가능성** | 실시간 번역 시스템, 커뮤니케이션 도구 | ❌ 번역과 관련된 응용이 아님 | QA 시스템, fact-checking, 지식기반 챗봇, 법률/의학 정보 응답 등으로 바꿔야 함 |

---

## 🔍 핵심 요약

- ❗ **주제를 완전히 오해한 요약**
- ❗ **논문을 읽지 않은 사람이 이해하기 위한 설명이 아님**
- ⚠️ 이런 경우, 사용자(=논문을 모르는 사람)는 **잘못된 정보를 진짜처럼 받아들일 위험** 있음

---

## ✅ 제대로 된 개선 방향

**요약 요청 프롬프트**는 다음 조건을 만족해야 합니다:

1. ❌ 추측하지 말고 context 기반으로만 답하라  
2. 📎 논문 제목, 초록, 도입부를 명확하게 포함  
3. 🧠 “이 논문은 기계 번역과 관계 없다”고 명시  
4. 📋 구조화된 템플릿으로 요청 (타이틀/목적/기여 등)  
5. 🧒 초심자도 이해할 수 있게 작성


대안으로 Hugging Face의 **`blossom` 모델**을 사용하는 방식으로 논문 요약 RAG 파이프라인을 구현해보겠습니다.  


---

## ✅ 요구사항 정리

- 모델: Hugging Face `blossom` (예: `blossom-mini` or `blossom-chat` 모델군)
- 기능:
  - 논문 URL에서 PDF 직접 파싱
  - LangChain-style 문서 분할
  - RAG처럼 간단한 context 검색 (embedding + retrieval)
  - 구조화된 프롬프트로 요약 요청
- 목적: "논문을 모르는 사용자가 이해할 수 있는 요약"

---

## ✅ Hugging Face 모델 기반 요약 파이프라인 (blossom)


In [64]:
# 📦 필요 패키지 설치
!pip install transformers sentence-transformers PyMuPDF faiss-cpu

# 🔧 설정
import requests, fitz
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1️⃣ PDF 불러오기 (메모리에서 처리)
url = "https://arxiv.org/pdf/2005.11401"
response = requests.get(url)
doc = fitz.open(stream=response.content, filetype="pdf")
full_text = "\n".join([page.get_text() for page in doc])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
from huggingface_hub import login

input_hg_token = input(f"hf_your_token_here")
login(token=input_hg_token)


### 📚 2️⃣ 문서 분할 및 벡터화

In [66]:
# 문단 단위로 쪼갬
chunks = [c.strip() for c in full_text.split('\n\n') if len(c.strip()) > 200]

# 문장 임베딩 생성 (Blossom 기반)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # blossom과 호환 가능
embeddings = embedder.encode(chunks, convert_to_tensor=False)

# 벡터 인덱스 생성
index = faiss.IndexFlatL2(embeddings[0].shape[0])
index.add(np.array(embeddings))


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 🔍 3️⃣ 유사한 context 검색

In [67]:

def retrieve_relevant_chunks(query, top_k=5):
    query_embedding = embedder.encode([query])[0]
    _, indices = index.search(np.array([query_embedding]), top_k)
    return [chunks[i] for i in indices[0]]

query = "논문의 핵심 내용과 기여점을 요약해줘"
retrieved_chunks = retrieve_relevant_chunks(query)
context = "\n\n".join(retrieved_chunks)


### Blossom 모델로 요약 요청

In [ ]:
# blossom 기반 모델 로딩
qa_pipe = pipeline("text-generation", model="mosaicml/blossom-mini", tokenizer="mosaicml/blossom-mini", max_new_tokens=512)

# 구조화 프롬프트 작성
structured_prompt = f"""
다음은 논문 요약을 위한 프롬프트입니다. 아래 내용을 참고해 템플릿에 맞게 요약해주세요.

--- 논문 내용 ---
{context}

--- 요약 형식 ---
1. 📌 제목 요약 (Title Summary):
2. 🧪 연구 목적 (Research Objective):
3. 🛠 방법론 요약 (Methodology):
4. 🔬 주요 실험 결과 (Key Findings):
5. 🧠 기여 및 의의 (Contribution):
6. 🌍 응용 가능성 (Applications):

"""
output = qa_pipe(structured_prompt)[0]['generated_text']
print(output)